# nflverse Weekly Stats - Automated Update

Automatically fetches the latest week's NFL stats from nflverse using the `nfl_data_py` Python package.

**Schedule:** Runs weekly on Tuesdays at 8:00 AM (America/Chicago)

**Features:**
- ✅ Free, open-source NFL data
- ✅ No API key required
- ✅ Auto-detects current NFL season and week
- ✅ Only fetches new data we don't already have
- ✅ 53 comprehensive stat fields per player

**Source:** nflverse.com via nfl_data_py Python package

In [0]:
# Install nfl_data_py package
%pip install nfl_data_py --quiet

import nfl_data_py as nfl
import pandas as pd
import json
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from datetime import datetime

print("✓ nfl_data_py installed")
print(f"✓ Current time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [0]:
# Historical backfill for all weeks in 2024 and 2025
# Run this ONCE to populate historical data

import time

print("="*70)
print("NFLVERSE - HISTORICAL BACKFILL")
print("="*70)
print("\nThis will fetch all available weeks for 2024 and 2025.")
print("⚠️  This is a one-time backfill - use the 'Fetch Latest Week Data' cell for ongoing updates\n")

# Define seasons and weeks to backfill
backfill_config = [
    {"season": 2024, "weeks": range(1, 19)},  # Regular season weeks 1-18
    {"season": 2025, "weeks": range(1, 19)}   # Regular season weeks 1-18
]

total_records = 0
successful_weeks = []
failed_weeks = []

for config in backfill_config:
    season = config["season"]
    weeks = config["weeks"]
    
    print(f"\n{'='*70}")
    print(f"Processing Season {season}")
    print(f"{'='*70}")
    
    # Fetch all weekly data for this season at once (nflverse is efficient)
    print(f"\nFetching nflverse data for entire {season} season...")
    
    try:
        weekly_stats_pd = nfl.import_weekly_data([season])
        print(f"✓ Downloaded {len(weekly_stats_pd)} total records for {season}")
    except Exception as e:
        print(f"❌ Error fetching {season} data: {e}")
        continue
    
    for week in weeks:
        try:
            print(f"\n[Week {week}] Processing...")
            
            # Filter for this specific week
            week_data = weekly_stats_pd[weekly_stats_pd['week'] == week]
            
            if len(week_data) == 0:
                print(f"  ⚠️  No data available - skipping")
                failed_weeks.append((season, week, "No data"))
                continue
            
            print(f"  Found {len(week_data)} player records")
            
            # Convert to Spark DataFrame
            rows = []
            for idx, row in week_data.iterrows():
                player_id = str(row.get('player_id', row.get('player_name', '')))
                fantasy_points = float(row.get('fantasy_points_ppr', 0) or 0)
                
                # Store all stats as JSON
                stats = row.to_dict()
                stats_clean = {}
                for k, v in stats.items():
                    if pd.isna(v):
                        stats_clean[k] = None
                    elif isinstance(v, (pd.Timestamp, datetime)):
                        stats_clean[k] = str(v)
                    else:
                        try:
                            stats_clean[k] = float(v) if isinstance(v, (int, float)) else str(v)
                        except:
                            stats_clean[k] = str(v)
                
                rows.append(
                    Row(
                        player_id=player_id,
                        week=int(week),
                        season=int(season),
                        fantasy_points=fantasy_points,
                        stats=json.dumps(stats_clean),
                        source='nflverse'
                    )
                )
            
            if not rows:
                print(f"  ⚠️  No valid player data - skipping")
                failed_weeks.append((season, week, "No valid data"))
                continue
            
            stats_df = spark.createDataFrame(rows)
            
            # Write to bronze
            bronze_df = stats_df.withColumn("ingested_at", F.current_timestamp())
            bronze_df.createOrReplaceTempView("nflverse_backfill_bronze")
            
            spark.sql("""
                MERGE INTO main.fantasai.bronze_weekly_stats AS target
                USING nflverse_backfill_bronze AS source
                ON target.player_id = source.player_id 
                    AND target.week = source.week 
                    AND target.season = source.season
                    AND target.source = source.source
                WHEN MATCHED THEN
                    UPDATE SET
                        target.fantasy_points = source.fantasy_points,
                        target.stats = source.stats,
                        target.ingested_at = source.ingested_at
                WHEN NOT MATCHED THEN
                    INSERT (player_id, week, season, fantasy_points, stats, source, ingested_at)
                    VALUES (source.player_id, source.week, source.season, source.fantasy_points, source.stats, source.source, source.ingested_at)
            """)
            
            # Write to silver
            silver_df = bronze_df.dropDuplicates(["player_id", "week", "season", "source"])
            silver_df.createOrReplaceTempView("nflverse_backfill_silver")
            
            spark.sql("""
                MERGE INTO main.fantasai.silver_weekly_stats AS target
                USING nflverse_backfill_silver AS source
                ON target.player_id = source.player_id 
                    AND target.week = source.week 
                    AND target.season = source.season
                    AND target.source = source.source
                WHEN MATCHED THEN
                    UPDATE SET
                        target.fantasy_points = source.fantasy_points,
                        target.stats = source.stats,
                        target.ingested_at = source.ingested_at
                WHEN NOT MATCHED THEN
                    INSERT (player_id, week, season, fantasy_points, stats, source, ingested_at)
                    VALUES (source.player_id, source.week, source.season, source.fantasy_points, source.stats, source.source, source.ingested_at)
            """)
            
            player_count = bronze_df.count()
            total_records += player_count
            successful_weeks.append((season, week))
            
            print(f"  ✓ Stored {player_count} unique players")
            
        except Exception as e:
            print(f"  ❌ Error: {e}")
            failed_weeks.append((season, week, str(e)))
            continue

print("\n" + "="*70)
print("BACKFILL COMPLETE")
print("="*70)
print(f"\n✓ Successfully processed {len(successful_weeks)} weeks")
print(f"✓ Total unique players stored: {total_records}")

if failed_weeks:
    print(f"\n⚠️  Failed weeks ({len(failed_weeks)}):")
    for season, week, reason in failed_weeks[:10]:
        print(f"  - {season} Week {week}: {reason}")
    if len(failed_weeks) > 10:
        print(f"  ... and {len(failed_weeks) - 10} more")

print("\n💡 Next: Use the 'Fetch Latest Week Data' cell for ongoing updates")

In [0]:
# Auto-detect current season and find latest week we DON'T have yet
print("="*70)
print("NFLVERSE - LATEST DATA FETCH")
print("="*70)
print(f"\nCurrent date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# Determine current season
current_date = datetime.now()
current_year = current_date.year
current_month = current_date.month

if current_month >= 9:  # September onwards = current year's season
    CURRENT_SEASON = current_year
else:  # January-August = previous year's season
    CURRENT_SEASON = current_year - 1

print(f"Detected NFL season: {CURRENT_SEASON}")

# Check what weeks we already have in the database for this season
print("\nChecking existing data in database...")
existing_weeks = spark.sql(f"""
    SELECT DISTINCT week 
    FROM main.fantasai.silver_weekly_stats 
    WHERE season = {CURRENT_SEASON} AND source = 'nflverse'
    ORDER BY week
""").collect()

existing_week_numbers = [row['week'] for row in existing_weeks]

if existing_week_numbers:
    print(f"Already have nflverse data for weeks: {existing_week_numbers}")
    # Find the next week we need
    WEEK = max(existing_week_numbers) + 1 if existing_week_numbers else 1
else:
    print("No nflverse data found for this season yet")
    WEEK = 1

# Don't go past week 18 (regular season)
if WEEK > 18:
    print(f"\n⚠️  Week {WEEK} exceeds regular season (weeks 1-18)")
    print("All weeks are already up to date!")
    dbutils.notebook.exit("No new data to fetch - all weeks current")

print(f"\nWill fetch: Season {CURRENT_SEASON}, Week {WEEK}")

try:
    # Fetch weekly player statistics
    print(f"\nFetching nflverse data for Week {WEEK}...")
    
    # Import weekly stats for the season
    weekly_stats_pd = nfl.import_weekly_data([CURRENT_SEASON])
    
    # Filter for the specific week
    weekly_stats_pd = weekly_stats_pd[weekly_stats_pd['week'] == WEEK]
    
    if len(weekly_stats_pd) == 0:
        print(f"\n⚠️  No data available yet for Week {WEEK}")
        print("Games may not have been played or data not yet published")
        dbutils.notebook.exit(f"No data available for Season {CURRENT_SEASON} Week {WEEK}")
    
    print(f"✓ Fetched {len(weekly_stats_pd)} player records\n")
    
    # Convert pandas DataFrame to Spark DataFrame
    rows = []
    for idx, row in weekly_stats_pd.iterrows():
        player_id = str(row.get('player_id', row.get('player_name', '')))
        fantasy_points = float(row.get('fantasy_points_ppr', 0) or 0)
        
        # Store all stats as JSON
        stats = row.to_dict()
        # Convert numpy types to Python types for JSON serialization
        stats_clean = {}
        for k, v in stats.items():
            if pd.isna(v):
                stats_clean[k] = None
            elif isinstance(v, (pd.Timestamp, datetime)):
                stats_clean[k] = str(v)
            else:
                try:
                    stats_clean[k] = float(v) if isinstance(v, (int, float)) else str(v)
                except:
                    stats_clean[k] = str(v)
        
        rows.append(
            Row(
                player_id=player_id,
                week=int(WEEK),
                season=int(CURRENT_SEASON),
                fantasy_points=fantasy_points,
                stats=json.dumps(stats_clean),
                source='nflverse'
            )
        )
    
    stats_df = spark.createDataFrame(rows)
    
    # Write to bronze
    bronze_df = stats_df.withColumn("ingested_at", F.current_timestamp())
    bronze_df.createOrReplaceTempView("nflverse_bronze_temp")
    
    spark.sql("""
        MERGE INTO main.fantasai.bronze_weekly_stats AS target
        USING nflverse_bronze_temp AS source
        ON target.player_id = source.player_id 
            AND target.week = source.week 
            AND target.season = source.season
            AND target.source = source.source
        WHEN MATCHED THEN
            UPDATE SET
                target.fantasy_points = source.fantasy_points,
                target.stats = source.stats,
                target.ingested_at = source.ingested_at
        WHEN NOT MATCHED THEN
            INSERT (player_id, week, season, fantasy_points, stats, source, ingested_at)
            VALUES (source.player_id, source.week, source.season, source.fantasy_points, source.stats, source.source, source.ingested_at)
    """)
    
    # Write to silver
    silver_df = bronze_df.dropDuplicates(["player_id", "week", "season", "source"])
    silver_df.createOrReplaceTempView("nflverse_silver_temp")
    
    spark.sql("""
        MERGE INTO main.fantasai.silver_weekly_stats AS target
        USING nflverse_silver_temp AS source
        ON target.player_id = source.player_id 
            AND target.week = source.week 
            AND target.season = source.season
            AND target.source = source.source
        WHEN MATCHED THEN
            UPDATE SET
                target.fantasy_points = source.fantasy_points,
                target.stats = source.stats,
                target.ingested_at = source.ingested_at
        WHEN NOT MATCHED THEN
            INSERT (player_id, week, season, fantasy_points, stats, source, ingested_at)
            VALUES (source.player_id, source.week, source.season, source.fantasy_points, source.stats, source.source, source.ingested_at)
    """)
    
    player_count = bronze_df.count()
    
    print("="*70)
    print("FETCH COMPLETE")
    print("="*70)
    print(f"\n✓ Season: {CURRENT_SEASON}")
    print(f"✓ Week: {WEEK}")
    print(f"✓ Players stored: {player_count}")
    print(f"\n📅 Data ingested at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
except Exception as e:
    print(f"\n❌ Error: {e}")
    import traceback
    traceback.print_exc()
    raise